In [ ]:
# TODO
#   1. Figure out a RAG approach
#   2. Figure out guardrails.

## RAG approach
1. Dense/vector search for semantic similarity
2. BM25/keyword search for exact medical terms
3. Metadata filtering for trusted sources/topic/date
4. Reranking to choose the strongest evidence
5. Evidence-grounded generation where the LLM is explicitly restricted to retrieved material

Original query: “I saw on TikTok that taking 10 mg of melatonin every night fixes sleep deprivation. Is that true?”
       ↓
Claim extraction
       ↓
"10 mg melatonin every night fixes sleep deprivation"
       ↓
Query rewriting
       ↓
1. melatonin effectiveness for sleep deprivation
2. melatonin dosage young adults
3. melatonin long-term safety
       ↓
Hybrid retrieval
   ┌───────────────┐
   │ Dense search  │
   │ BM25 search   │
   └──────┬────────┘
          ↓
Trusted-source filter
          ↓
Reranker
          ↓
Top 3–5 evidence chunks
          ↓
Evidence sufficiency check
          ↓
LLM
          ↓
Grounded educational response + citations

## Overall structure for the project.

                    USER
                      │
                      ▼
             Text / Image Input
                      │
                      ▼
              INPUT GUARDRAIL
         privacy / safety / scope
                      │
                      ▼
              CLAIM EXTRACTION
                      │
                      ▼
           QUERY CLASSIFICATION
                     /              
                Supplements          
                    |       
                    │
                    ▼
              QUERY REWRITING
                    │
                    ▼
        ┌───────────────────────┐
        │      HYBRID RAG       │
        │                       │
        │ Dense vector search   │
        │          +            │
        │ BM25 keyword search   │
        └───────────┬───────────┘
                    │
                    ▼
           SOURCE/METADATA FILTER
                    │
                    ▼
                RERANKER
                    │
                    ▼
          EVIDENCE SUFFICIENT?
              /           \
            No             Yes
            │               │
      Rewrite/search        ▼
         again       EVIDENCE PACKAGE
                            │
                            ▼
                           LLM
                            │
                            ▼
                    OUTPUT GUARDRAIL
                   /        |        \
             grounded?   safe?    citations?
                            │
                            ▼
                  STRUCTURED RESPONSE
                  • claim assessment
                  • explanation
                  • evidence strength
                  • uncertainty
                  • sources
                            │
                            ▼
                           USER

## RAG technique:
Use Corrective RAG.

Retrieve evidence
       ↓
Evaluate retrieval quality
       ↓
Is the evidence good?
   /            \
 Yes            No
 │               │
Use it      Rewrite query
 │               │
 │          Retrieve again
 │               │
 └───────┬───────┘
         ▼
     Generate

## Imports + Model prepping

In [ ]:
import os
from pathlib import Path
import pandas as pd
import chromadb
from openai import OpenAI

In [ ]:
client = OpenAI(
    api_key=os.environ["COMP3110_API_KEY"],
    base_url=os.environ["COMP3110_MODEL_BASE_URL"]
)

MODEL = "scx-gpt-oss-120b"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"


## Grab Documents


In [ ]:
# TODO grab at least 10 trusted documents about supplements
# Try to do each document in this structure: id, title, source, url, topic, text.

documents = []

In [ ]:
documents_df = pd.DataFrame(documents)
documents_df[["id", "title", "source", "topic"]]

## Chunk Documents

In [ ]:
def chunk_text(
    text, 
    chunk_size=300,
    overlap=50
):
    chunks = []
    words = text.split()
    start = 0

    while start < len(words):
        chunk = words[start:start + chunk_size]
        chunks.append(" ".join(chunk))
        start += chunk_size - overlap
    return chunks

In [ ]:
chunks = []

for document in documents:

    document_chunks = chunk_text(
        document["text"]
    )

    for index, text in enumerate(
        document_chunks
    ):

        chunks.append(
            {
                "id": f"{document['id']}_{index}",
                "text": text,
                "title": document["title"],
                "source": document["source"],
                "url": document["url"],
                "topic": document["topic"],
            }
        )

In [ ]:
chunks_df = pd.DataFrame(chunks)
chunks_df.head()

print(f"Documents: {len(documents)}")
print(f"Chunks: {len(chunks)}")

## Embeddings


In [ ]:
def create_embedding(text):
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )

    return response.data[0].embedding

In [ ]:
# Test embedding
test_embedding = create_embedding("Does magnesium improve sleep?")

print(len(test_embedding))

## Vector Database

In [ ]:
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="medical_evidence")

## Embed and store the chunks

In [ ]:
for chunk in chunks:

    embedding = create_embedding(chunk["text"])

    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[
            {
                "title": chunk["title"],
                "source": chunk["source"],
                "url": chunk["url"],
                "topic": chunk["topic"],
            }
        ],
    )

# Check collection
collection.count()

## Testing Vector Retrieval

In [ ]:
def vector_search(query, top_k=5,):
    query_embedding = create_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
    )

    return results

# Testing
results = vector_search("Can magnesium help me sleep?")
results

## Retrieval inspection


In [ ]:
def show_results(results):

    rows = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for text, metadata, distance in zip(
        documents,
        metadatas,
        distances,
    ):
        rows.append(
            {
                "source": metadata["source"],
                "title": metadata["title"],
                "topic": metadata["topic"],
                "distance": distance,
                "text": text,
            }
        )

    return pd.DataFrame(rows)

show_results(results)

## Topic filtering

In [ ]:
def vector_search(
    query,
    topic=None,
    top_k=5,
):

    query_embedding = create_embedding(query)

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
    }

    if topic is not None:
        query_args["where"] = {
            "topic": topic
        }

    return collection.query(**query_args)

## Testing
results = vector_search(
    query="Does magnesium help sleep?",
    topic="supplements",
)

show_results(results)

## Claim Extraction

In [ ]:
def extract_claims(user_input):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
You extract medical claims from user messages.

Return each independent factual medical claim on a separate line.
Do not evaluate whether the claim is correct.
"""
            },
            {
                "role": "user",
                "content": user_input,
            },
        ],
        temperature=0,
    )

    output = (
        response
        .choices[0]
        .message
        .content
    )

    claims = [
        claim.strip("- ").strip()
        for claim in output.splitlines()
        if claim.strip()
    ]

    return claims

# Testing
user_input = """
    My friend said magnesium glycinate makes you fall asleep
    because it increases melatonin.
"""
claims = extract_claims(user_input)
claims

## Query Rewriting

In [ ]:
def rewrite_query(claim):

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": """
Convert medical claims into concise evidence-search queries.

Return only the search query.
"""
            },
            {
                "role": "user",
                "content": claim,
            },
        ],
        temperature=0,
    )

    return (
        response
        .choices[0]
        .message
        .content
        .strip()
    )

for claim in claims:
    query = rewrite_query(claim)
    print("Claim:", claim)
    print("Query:", query)
    print()

## Evidence retrieval

In [ ]:
all_evidence = []

for claim in claims:

    query = rewrite_query(claim)
    results = vector_search(
        query=query,
        top_k=5,
    )
    evidence_df = show_results(results)

    evidence_df["claim"] = claim
    evidence_df["query"] = query

    all_evidence.append(evidence_df)

In [ ]:
# Combine
evidence_df = pd.concat(
    all_evidence,
    ignore_index=True,
)
evidence_df

## Context Formatting

In [ ]:
def format_context(results):

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]

    sections = []

    for i, (text, metadata) in enumerate(
        zip(documents, metadatas),
        start=1,
    ):

        section = f"""
SOURCE {i}

Title:
{metadata["title"]}

Organisation:
{metadata["source"]}

URL:
{metadata["url"]}

Evidence:
{text}
"""

        sections.append(section)

    return "\n".join(
        sections
    )

In [ ]:
# Testing
results = vector_search(
    "magnesium supplementation sleep quality"
)
context = format_context(results)
print(context)

## RAG Generation

In [ ]:
def generate_answer(
    question,
    context,
):

    system_prompt = """
You are an educational medical misinformation assistant aimed
primarily at young adults.

Your purpose is to explain medical evidence, not provide personalised
medical advice.

Rules:
- Use the retrieved evidence to support factual medical claims.
- Do not diagnose the user.
- Do not prescribe treatment.
- Clearly communicate uncertainty.
- Do not invent evidence.
- If the retrieved evidence is insufficient, say so.
- Cite evidence using [Source 1], [Source 2], etc.
- Explain unfamiliar medical terminology using simple language.

Use this format:

Claim assessment:
Short answer:
What the evidence says:
Important context:
Evidence confidence:
"""

    user_prompt = f"""
USER QUESTION

{question}


RETRIEVED EVIDENCE

{context}
"""

    response = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],

        temperature=0.1,
    )

    return (
        response
        .choices[0]
        .message
        .content
    )

In [ ]:
# Testing
answer = generate_answer(
    question=(
        "Does magnesium really help you sleep?"
    ),
    context=context,
)

print(answer)

## RAG function

In [ ]:
def rag_answer(
    user_input,
    top_k=5,
):

    results = vector_search(
        query=user_input,
        top_k=top_k,
    )

    context = format_context(results)

    answer = generate_answer(
        question=user_input,
        context=context,
    )

    return {
        "answer": answer,
        "results": results,
    }

In [ ]:
# Testing
result = rag_answer("Does melatonin fix sleep deprivation?")
print(result["answer"])

## Claim-aware pipeline

In [ ]:
def claim_rag_answer(
    user_input,
    top_k=5,
):

    claims = extract_claims(user_input)

    retrieved_results = []

    for claim in claims:

        query = rewrite_query(claim)

        results = vector_search(
            query=query,
            top_k=top_k,
        )

        retrieved_results.append(
            {
                "claim": claim,
                "query": query,
                "results": results,
            }
        )

    return retrieved_results

In [ ]:
# Testing
result = claim_rag_answer(
    """
    Taking melatonin means you can sleep four hours
    a night without suffering from sleep deprivation.
    """
)

result

## Experimentation / Evaluation

In [ ]:
test_questions = [
    "Does magnesium improve sleep?",
    "Can melatonin replace lost sleep?",
    "Can supplements prevent the effects of sleep deprivation?",
    "Is taking more melatonin always better?",
    "Can four hours of sleep be enough if I take supplements?",
]

In [ ]:
for question in test_questions:

    print("=" * 80)
    print("QUESTION:")
    print(question)

    result = rag_answer(question)

    print("\nANSWER:")
    print(result["answer"])
    print()